<a href="https://www.kaggle.com/code/murtazaabdullah2010/what-s-the-buzz-private-1-0000-score?scriptVersionId=368866639" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/cyprus-summer-ai-camp-2026-tasks/sample_submission.csv
/kaggle/input/competitions/cyprus-summer-ai-camp-2026-tasks/train.csv
/kaggle/input/competitions/cyprus-summer-ai-camp-2026-tasks/ioai_cyprus_2026_local_setup.md
/kaggle/input/competitions/cyprus-summer-ai-camp-2026-tasks/test.csv


In [18]:
train_ds= pd.read_csv("/kaggle/input/competitions/cyprus-summer-ai-camp-2026-tasks/train.csv")
test_ds= pd.read_csv("/kaggle/input/competitions/cyprus-summer-ai-camp-2026-tasks/test.csv")
sample_sub = pd.read_csv("/kaggle/input/competitions/cyprus-summer-ai-camp-2026-tasks/sample_submission.csv")

In [4]:
train_ds

,class,name,text,description
0,0,Personal Finance,I set up a monthly budget and moved part of my...,"Managing your own money: budgeting, saving, pa..."
1,1,Startups & Business,The founders raised a seed round and used the ...,"Building and running companies: founders, fund..."
2,2,Travel & Tourism,We booked a flight and a small hotel near the ...,"Trips and destinations: flights, hotels, sight..."
3,3,Food & Dining,I simmered the sauce for an hour and served th...,"Cooking and eating: recipes, restaurants, ingr..."
4,4,Space & Astronomy,"The telescope captured a distant galaxy, revea...","Stars, planets, telescopes, space missions and..."
5,5,Gadgets & Consumer Tech,"The new phone has a faster chip, a brighter sc...","Consumer electronics: phones, laptops, headpho..."


In [6]:
train_ds["text"] = train_ds["text"] +train_ds["description"]

In [27]:
print("train_cols: ",train_ds.columns)
print("Test_cols: ", test_ds.columns)

train_cols:  Index(['class', 'name', 'text', 'description'], dtype='object')
Test_cols:  Index(['id', 'text'], dtype='object')


In [30]:
train= train_ds
test= test_ds
##the whole code was written by gpt as i was supper lazy but idea was mine lol
import os
import json
import re
import time
import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key="your_key"
)

MODEL = "nvidia/nemotron-3.5-super-vl:free"

classes = []

for _, row in train.iterrows():
    classes.append({
        "class": str(row["class"]),
        "name": str(row["name"]),
        "description": str(row["description"]),
        "reference_example": str(row["text"])
    })

class_labels = [c["class"] for c in classes]

SYSTEM_PROMPT = """
You are an expert semantic classification system.

Your task is to classify short texts into exactly one of a
fixed set of six semantic categories.

You have access to three types of information for every class:

1. CLASS NAME:
   The name or title of the category.

2. CLASS DESCRIPTION:
   A detailed explanation of the topics, concepts, subject
   matter, and semantic boundaries belonging to this category.

3. REFERENCE EXAMPLE:
   One real example of text belonging to this category.

IMPORTANT CLASSIFICATION STRATEGY:

- Treat the class description as the primary definition
  of the category.
- Treat the reference example as a demonstration of how
  the category manifests in actual text.
- Do not simply match keywords.
- Understand the underlying meaning, intent, context,
  and subject matter of the input.
- A test text does not need to resemble its reference
  example lexically.
- Use the descriptions to generalize beyond the single
  available example.
- Consider all six classes before deciding.
- Distinguish between closely related categories carefully.
- Do not assume that a shared keyword means shared class.
- Prefer the category whose overall semantic definition
  best explains the input.

DECISION PROCESS:

1. Understand the main subject and intent of the input.
2. Identify its most important concepts.
3. Compare those concepts against all six descriptions.
4. Use the reference examples to resolve ambiguity.
5. Select the single most semantically appropriate class.

Do not force an uncertain classification based on
superficial similarities.

Return valid JSON only.
"""

def build_prompt(text):

    class_info = json.dumps(
        classes,
        ensure_ascii=False,
        indent=2
    )

    return f"""
Here are all available classes and their information:

{class_info}

Now classify the following text.

<INPUT_TEXT>
{text}
</INPUT_TEXT>

Return exactly this JSON structure:

{{
    "predicted_class": "exact class label",
    "confidence": 0.0,
    "reasoning": "Brief explanation of why this class fits better than the alternatives."
}}

The predicted_class MUST be one of these labels:

{class_labels}

Confidence must be between 0 and 1.

Do not output anything outside the JSON.
"""


def classify(text, retries=3):

    prompt = build_prompt(str(text))

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.1,
                max_tokens=500,
                response_format={
                    "type": "json_object"
                }
            )

            content = response.choices[0].message.content

            result = json.loads(content)

            prediction = str(result["predicted_class"])

            if prediction not in class_labels:
                raise ValueError(
                    f"Invalid class: {prediction}"
                )

            return {
                "predicted_class": prediction,
                "confidence": float(result["confidence"]),
                "reasoning": result["reasoning"]
            }

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(2 ** attempt)

    raise RuntimeError("Classification failed after retries")


results = []

for _, row in tqdm(test.iterrows(), total=len(test)):

    result = classify(row["text"])

    results.append({
        "id": row["id"],
        "class": result["predicted_class"],
        "confidence": result["confidence"],
        "reasoning": result["reasoning"]
    })


predictions = pd.DataFrame(results)

# Restore original class data types
class_mapping = {
    str(c): c for c in train["class"].unique()
}

predictions["class"] = predictions["class"].map(class_mapping)

predictions.to_csv("llm_predictions.csv", index=False)

submission = predictions[["id", "class"]]
submission.to_csv("submission.csv", index=False)

print(submission.head())
print(predictions["class"].value_counts())
print("Saved submission.csv")


  0%|          | 0/36 [00:00<?, ?it/s]

Attempt 1 failed: the JSON object must be str, bytes or bytearray, not NoneType
Attempt 1 failed: Unterminated string starting at: line 1 column 59 (char 58)
Attempt 1 failed: Unterminated string starting at: line 1 column 59 (char 58)
Attempt 1 failed: Unterminated string starting at: line 1 column 59 (char 58)
Attempt 1 failed: Unterminated string starting at: line 1 column 59 (char 58)
        id  class
0  msg_000      4
1  msg_001      1
2  msg_002      4
3  msg_003      0
4  msg_004      4
class
4    6
1    6
0    6
5    6
3    6
2    6
Name: count, dtype: int64
Saved submission.csv
